# 📊 Exploración de la Inflación Colombiana (2018–2024)

**Autor:** Brausin  
**Fecha:** 2024  
**Fuentes:** DANE, Banco de la República de Colombia

---

## Objetivos

En este notebook analizamos la evolución histórica de la inflación en Colombia entre 2018 y 2024, con énfasis en:

1. El **impacto de la pandemia COVID-19** en los precios (2020)
2. El **pico inflacionario de 2022** — el más alto en 24 años
3. El proceso de **desinflación gradual en 2023–2024**
4. La relación entre la **tasa de política monetaria** y la inflación

> 💡 **Contexto:** Colombia experimentó en 2022 una inflación del 13.12% anual, el nivel más alto desde 1998. Este fenómeno fue impulsado por choques de oferta globales (guerra en Ucrania, disrupciones logísticas), la recuperación acelerada de la demanda post-pandemia, y factores internos como el paro nacional de 2021.

In [ ]:
# Importaciones
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

print('✅ Librerías cargadas correctamente')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')

## 1. Carga de datos

Utilizamos el módulo `colombia_data` del repositorio para obtener la serie histórica del IPC y la tasa de intervención del Banco de la República. Los datos corresponden a cifras oficiales publicadas mensualmente.

In [ ]:
import sys
sys.path.insert(0, '../src')

from colombia_data import BancoRepublicaClient, DANEClient

# Instanciar clientes
banrep = BancoRepublicaClient()
dane = DANEClient()

# Obtener datos
df_ipc = banrep.obtener_ipc(anio_inicio=2018, anio_fin=2024)
df_tasa = banrep.obtener_tasa_intervencion()
df_desempleo = dane.obtener_desempleo(anio_inicio=2018, anio_fin=2024)

print(f'IPC: {len(df_ipc)} registros mensuales')
print(f'Tasa intervención: {len(df_tasa)} cambios')
print(f'Desempleo: {len(df_desempleo)} trimestres')
df_ipc.head()

## 2. Estadísticas descriptivas

Antes de visualizar, exploramos las estadísticas básicas de la serie de inflación para identificar rangos, promedios y puntos extremos.

In [ ]:
print('=== Estadísticas descriptivas: IPC (variación anual %) ===')
print(df_ipc['variacion_anual'].describe().round(2))
print()

# Máximos y mínimos
idx_max = df_ipc['variacion_anual'].idxmax()
idx_min = df_ipc['variacion_anual'].idxmin()

print(f'📈 Máximo: {df_ipc.loc[idx_max, "variacion_anual"]}% '
      f'en {df_ipc.loc[idx_max, "fecha"].strftime("%B %Y")}')
print(f'📉 Mínimo: {df_ipc.loc[idx_min, "variacion_anual"]}% '
      f'en {df_ipc.loc[idx_min, "fecha"].strftime("%B %Y")}')
print()

# Promedio por año
print('=== Promedio de inflación por año ===')
print(df_ipc.groupby('anio')['variacion_anual'].mean().round(2).to_string())

## 3. Visualización principal: Evolución del IPC (2018–2024)

La siguiente gráfica muestra la trayectoria completa de la inflación colombiana. Se destacan tres períodos clave:

- 🟦 **2020**: Caída por COVID-19 — la demanda se desplomó, llevando la inflación a mínimos históricos recientes (1.49% en noviembre de 2020)
- 🟥 **2021–2022**: Recuperación explosiva y choque de precios — la inflación se disparó al 13.12% en diciembre de 2022
- 🟩 **2023–2024**: Desinflación — el Banco de la República subió las tasas de interés al 13.25% para controlar la inflación

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Línea principal
ax.plot(df_ipc['fecha'], df_ipc['variacion_anual'],
        color='#E63946', linewidth=2.5, zorder=3, label='IPC (variación anual %)')

# Relleno bajo la curva
ax.fill_between(df_ipc['fecha'], df_ipc['variacion_anual'], alpha=0.15, color='#E63946')

# Meta de inflación del Banrep (3% ± 1pp)
ax.axhspan(2, 4, alpha=0.1, color='green', label='Meta inflación (2%–4%)')
ax.axhline(y=3, color='green', linestyle='--', linewidth=1, alpha=0.7)

# Anotar puntos clave
anotaciones = [
    ('2020-11-01', 1.49, 'Mínimo pandemia\n1.49%', 'bottom'),
    ('2022-12-01', 13.12, 'Máximo 2022\n13.12%', 'top'),
    ('2024-12-01', 5.02, 'Dic 2024\n5.02%', 'bottom'),
]

for fecha_str, valor, texto, va in anotaciones:
    fecha = pd.Timestamp(fecha_str)
    offset = 0.4 if va == 'top' else -0.4
    ax.annotate(texto,
                xy=(fecha, valor),
                xytext=(fecha, valor + offset * 3),
                fontsize=8.5, ha='center', va=va,
                color='#333333',
                arrowprops=dict(arrowstyle='->', color='#555555', lw=1.2),
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='#cccccc'))

# Formato de ejes
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
plt.xticks(rotation=45, ha='right')
ax.set_ylabel('Variación anual (%)', fontsize=11)
ax.set_title('Inflación en Colombia (IPC variación anual)\n2018 – 2024',
             fontsize=14, fontweight='bold', pad=15)
ax.legend(loc='upper left', framealpha=0.9)
ax.set_ylim(0, 15)

# Fuente
fig.text(0.99, 0.01, 'Fuente: DANE / Banco de la República de Colombia',
         ha='right', va='bottom', fontsize=8, color='#888888')

plt.tight_layout()
plt.savefig('../assets/01_ipc_historico.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfica guardada en assets/01_ipc_historico.png')

## 4. Inflación promedio anual

El siguiente gráfico de barras resume el promedio de inflación por año, permitiendo comparar de forma clara los períodos de baja y alta inflación.

In [ ]:
inflacion_anual = df_ipc.groupby('anio')['variacion_anual'].mean().reset_index()
inflacion_anual.columns = ['anio', 'promedio']

fig, ax = plt.subplots(figsize=(10, 5))

# Colores por nivel
colores = ['#2A9D8F' if v <= 4 else '#E9C46A' if v <= 8 else '#E63946'
           for v in inflacion_anual['promedio']]

bars = ax.bar(inflacion_anual['anio'].astype(str),
              inflacion_anual['promedio'],
              color=colores, edgecolor='white', linewidth=0.5, width=0.6)

# Etiquetas en las barras
for bar, val in zip(bars, inflacion_anual['promedio']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Meta de inflación
ax.axhline(y=3, color='#2A9D8F', linestyle='--', linewidth=1.5,
           label='Meta Banrep (3%)', alpha=0.8)

# Leyenda de colores
p1 = mpatches.Patch(color='#2A9D8F', label='Dentro de meta (≤4%)')
p2 = mpatches.Patch(color='#E9C46A', label='Elevada (4%–8%)')
p3 = mpatches.Patch(color='#E63946', label='Alta (>8%)')
ax.legend(handles=[p1, p2, p3], loc='upper left', framealpha=0.9, fontsize=9)

ax.set_title('Inflación promedio anual en Colombia\n2018 – 2024',
             fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Variación anual promedio (%)', fontsize=10)
ax.set_xlabel('Año', fontsize=10)
ax.set_ylim(0, 15)

fig.text(0.99, 0.01, 'Fuente: DANE / Banco de la República',
         ha='right', fontsize=8, color='#888888')

plt.tight_layout()
plt.savefig('../assets/02_inflacion_anual.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Tasa de intervención vs. Inflación

Una de las herramientas más importantes del Banco de la República es la **tasa de intervención** (tasa de política monetaria). Cuando la inflación sube, el banco sube la tasa para encarecer el crédito y enfriar la demanda.

Esta gráfica muestra cómo el Banco de la República respondió al pico inflacionario de 2022 con una de las subidas de tasas más agresivas de su historia.

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

# IPC en eje izquierdo
ax1.plot(df_ipc['fecha'], df_ipc['variacion_anual'],
         color='#E63946', linewidth=2.2, label='IPC (var. anual %)', zorder=3)
ax1.set_ylabel('IPC — Variación anual (%)', color='#E63946', fontsize=11)
ax1.tick_params(axis='y', labelcolor='#E63946')
ax1.set_ylim(0, 16)

# Tasa de intervención en eje derecho
ax2 = ax1.twinx()
ax2.step(df_tasa['fecha'], df_tasa['tasa_intervencion'],
         color='#1D3557', linewidth=2.2, label='Tasa intervención (%)',
         where='post', zorder=2)
ax2.fill_between(df_tasa['fecha'], df_tasa['tasa_intervencion'],
                 step='post', alpha=0.08, color='#1D3557')
ax2.set_ylabel('Tasa de intervención (%)', color='#1D3557', fontsize=11)
ax2.tick_params(axis='y', labelcolor='#1D3557')
ax2.set_ylim(0, 16)

# Formato de fechas
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
plt.xticks(rotation=45, ha='right')

# Leyenda combinada
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', framealpha=0.9)

ax1.set_title('Política Monetaria vs. Inflación en Colombia\nTasa de intervención y IPC (2020–2024)',
              fontsize=13, fontweight='bold', pad=12)

fig.text(0.99, 0.01, 'Fuente: Banco de la República de Colombia',
         ha='right', fontsize=8, color='#888888')

plt.tight_layout()
plt.savefig('../assets/03_tasa_vs_ipc.png', dpi=150, bbox_inches='tight')
plt.show()
print('La tasa de intervención subió de 1.75% (2021) a 13.25% (2023) para controlar la inflación')

## 6. Distribución mensual de la inflación

¿En qué meses tiende a ser más alta la inflación en Colombia? El siguiente gráfico analiza el patrón estacional del IPC.

In [ ]:
meses_nombres = {
    1: 'Ene', 2: 'Feb', 3: 'Mar', 4: 'Abr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Ago', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dic'
}

df_ipc['mes_nombre'] = df_ipc['mes'].map(meses_nombres)
promedio_mes = df_ipc.groupby('mes')['variacion_anual'].mean().reset_index()
promedio_mes['mes_nombre'] = promedio_mes['mes'].map(meses_nombres)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Boxplot por mes ---
ax = axes[0]
datos_mes = [df_ipc[df_ipc['mes'] == m]['variacion_anual'].values for m in range(1, 13)]
bp = ax.boxplot(datos_mes, labels=[meses_nombres[m] for m in range(1, 13)],
                patch_artist=True, medianprops=dict(color='#E63946', linewidth=2))
for patch in bp['boxes']:
    patch.set_facecolor('#AEC6CF')
    patch.set_alpha(0.7)
ax.set_title('Distribución del IPC por mes\n(2018–2024)', fontweight='bold')
ax.set_ylabel('Variación anual (%)')
ax.set_xlabel('Mes')

# --- Barras de promedio mensual ---
ax2 = axes[1]
ax2.bar(promedio_mes['mes_nombre'], promedio_mes['variacion_anual'],
        color='#457B9D', edgecolor='white', width=0.7)
for i, (_, row) in enumerate(promedio_mes.iterrows()):
    ax2.text(i, row['variacion_anual'] + 0.05, f"{row['variacion_anual']:.1f}%",
             ha='center', va='bottom', fontsize=8)
ax2.set_title('Promedio histórico del IPC por mes\n(2018–2024)', fontweight='bold')
ax2.set_ylabel('Variación anual promedio (%)')
ax2.set_xlabel('Mes')

fig.text(0.99, 0.01, 'Fuente: DANE / Banco de la República', ha='right', fontsize=8, color='#888888')
plt.tight_layout()
plt.savefig('../assets/04_distribucion_mensual.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Conclusiones

### Hallazgos principales

1. **El COVID-19 fue desinflacionario en 2020**: La contracción de la demanda llevó la inflación a mínimos históricos recientes (1.49% en noviembre 2020), muy por debajo de la meta del 3% del Banco de la República.

2. **2022: El peor año inflacionario en dos décadas**: La combinación de demanda reprimida post-pandemia, disrupciones de cadenas de suministro globales y el conflicto en Ucrania (que elevó precios de alimentos y energía) llevó la inflación al 13.12% anual — el nivel más alto desde 1998.

3. **Respuesta agresiva del Banco de la República**: En 2022–2023, el banco subió la tasa de intervención de 1.75% a 13.25%, una de las subidas más pronunciadas de su historia.

4. **Desinflación en curso**: Para 2024, la inflación cayó a niveles de 5–6%, aún por encima de la meta del 3% pero en clara trayectoria descendente.

### Próximos pasos

- Comparar la inflación colombiana con la regional (Chile, México, Brasil, Perú)
- Analizar la inflación por componentes: alimentos, energía, servicios
- Modelar la trayectoria esperada de desinflación para 2025
- Analizar el impacto en el tipo de cambio USD/COP

In [ ]:
# Resumen final en tabla
resumen = df_ipc.groupby('anio').agg(
    promedio=('variacion_anual', 'mean'),
    minimo=('variacion_anual', 'min'),
    maximo=('variacion_anual', 'max')
).round(2)

print('=== Resumen de inflación colombiana por año ===')
print(resumen.to_string())
print()
print('✅ Análisis completado')